# Model training Notebook

## Import dependancies

- os - pour manipuler les images.
- numpy - pour centrer et découper l'image en quatre
- PIL - pour charger les images
- sklearn - pour le model
- matplotlib - pour générer les graphiques

In [ ]:
import os
import numpy as np
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix
from sklearn.tree import export_text
import matplotlib.pyplot as plt

In [ ]:
def extract_clinical_features(arr, npix):
    center = [shape // 2 for shape in arr.shape]
    slices = tuple(slice(max(0, c - npix), c + npix) for c in center)
    cropped = arr[slices]
    
    h, w = cropped.shape
    half_h, half_w = h // 2, w // 2
    
    tl = cropped[:half_h, :half_w]
    tr = cropped[:half_h, half_w:]
    bl = cropped[half_h:, :half_w]
    br = cropped[half_h:, half_w:]
    
    features = [
        np.mean(tl), np.std(tl),
        np.mean(tr), np.std(tr),
        np.mean(bl), np.std(bl),
        np.mean(br), np.std(br)
    ]
    
    return features

In [ ]:
def load_images_from_folder(folder, label, npix=50):
    features_list = []
    labels = []
    
    if not os.path.exists(folder):
        print(f"Directory not found: {folder}")
        return features_list, labels

    print(f"Extracting clinical features from {folder}...")
    for filename in os.listdir(folder):
        path = os.path.join(folder, filename)
        if path.lower().endswith(('.png', '.jpg', '.jpeg')):
            try:
                img = Image.open(path).convert('L')
                arr = np.array(img)
                
                features = extract_clinical_features(arr, npix)
                features_list.append(features)
                labels.append(label)
            except Exception as e:
                print(f"Skipped {path}: {e}")
                
    return features_list, labels

In [ ]:
train_normal_dir = "../output/chest_xray/train/NORMAL"
train_pneu_dir = "../output/chest_xray/train/PNEUMONIA"
test_normal_dir = "../output/chest_xray/test/NORMAL"
test_pneu_dir = "../output/chest_xray/test/PNEUMONIA"

train_normal_x, train_normal_y = load_images_from_folder(train_normal_dir, 0)
train_pneu_x, train_pneu_y = load_images_from_folder(train_pneu_dir, 1)

In [ ]:
if len(train_normal_x) == 0 or len(train_pneu_x) == 0:
    raise ValueError("Train directories are empty or not found.")

X_train = np.array(train_normal_x + train_pneu_x)
y_train = np.array(train_normal_y + train_pneu_y)
print(f"\nTrain set loaded: {len(X_train)} samples.")

test_normal_x, test_normal_y = load_images_from_folder(test_normal_dir, 0)
test_pneu_x, test_pneu_y = load_images_from_folder(test_pneu_dir, 1)

if len(test_normal_x) == 0 or len(test_pneu_x) == 0:
    raise ValueError("Test directories are empty or not found.")

X_test = np.array(test_normal_x + test_pneu_x)
y_test = np.array(test_normal_y + test_pneu_y)
print(f"Test set loaded: {len(X_test)} samples.\n")

model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=50, 
    min_samples_leaf=1, 
    random_state=42, 
    class_weight="balanced"
)

print("Training model on the FULL train set...")
model.fit(X_train, y_train)
print("Model trained successfully.\n")

print("Evaluating model on the test set...")
y_proba = model.predict_proba(X_test)[:, 1]

threshold = 0.67
y_pred_strict = (y_proba >= threshold).astype(int)

print("\n--- RESULTS ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_strict):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred_strict))

if y_proba is not None:
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"ROC AUC: {roc_auc:.4f}")
        
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_strict))

feature_names = [
    "TL_Density", "TL_Heterogeneity",
    "TR_Density", "TR_Heterogeneity",
    "BL_Density", "BL_Heterogeneity",
    "BR_Density", "BR_Heterogeneity"
]

print("\n--- DECISION RULES (Sample from one tree) ---")
tree_rules = export_text(model.estimators_[0], feature_names=feature_names)
print(tree_rules)

if y_proba is not None:
    print("Generating plots...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    jitter = np.random.normal(0, 0.05, size=len(y_test))
    ax1.scatter(y_test + jitter, y_proba, alpha=0.5, color='royalblue')
    ax1.axhline(threshold, color='red', linestyle='--', label=f'Decision Threshold ({threshold})')
    ax1.set_xlabel('True Label (jittered for visibility)')
    ax1.set_ylabel('Predicted Probability of Pneumonia')
    ax1.set_title(f'Predictions (ROC AUC = {roc_auc:.4f})')
    ax1.set_xticks([0, 1])
    ax1.set_xticklabels(['Normal (0)', 'Pneumonia (1)'])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feature_names)))
    
    ax2.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx], color=colors)
    ax2.set_xlabel('Weight in Decision (%)')
    ax2.set_title('Clinical Features Importance (4 Quadrants)')
    
    plt.tight_layout()
    plt.show()